In [1]:
import pandas as pd
print(pd.__version__)

3.0.6


In [2]:
import pandas as pd

telco = pd.read_csv("../data/raw/telco_churn.csv")
bank = pd.read_csv("../data/raw/bank_churn.csv")
print(telco.shape, bank.shape)

(7043, 21) (10000, 14)


In [3]:
telco.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
telco.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
telco[telco["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "Churn"]]

,customerID,tenure,MonthlyCharges,Churn
488,4472-LVYGI,0,52.55,No
753,3115-CZMZD,0,20.25,No
936,5709-LVOEQ,0,80.85,No
1082,4367-NUYAO,0,25.75,No
1340,1371-DWPAZ,0,56.05,No
3331,7644-OMVMY,0,19.85,No
3826,3213-VVOLG,0,25.35,No
4380,2520-SGTTA,0,20.00,No
5218,2923-ARZLG,0,19.70,No
6670,4075-WKNIU,0,73.35,No


In [6]:
telco["TotalCharges"] = telco["TotalCharges"].fillna(0)
print(telco.isna().sum().sum())   # total missing values left in the whole table

0


In [7]:
print(telco["Churn"].value_counts())
telco["churn"] = (telco["Churn"] == "Yes").astype(int)
telco = telco.drop(columns=["Churn"])
print(telco["churn"].value_counts(normalize=True))

Churn
No     5174
Yes    1869
Name: count, dtype: int64
churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [8]:
cat_cols = telco.select_dtypes(exclude="number").columns
for c in cat_cols:
    print(c, telco[c].unique()[:6])

customerID <ArrowStringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC']
Length: 6, dtype: str
gender <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
Partner <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines <ArrowStringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService <ArrowStringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
OnlineSecurity <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup <ArrowStringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingTV <ArrowStringArray>
['No', 'Yes', 'No inter

In [9]:
print(pd.crosstab(telco["InternetService"], telco["OnlineSecurity"]))
print(pd.crosstab(telco["PhoneService"], telco["MultipleLines"]))

OnlineSecurity     No  No internet service   Yes
InternetService                                 
DSL              1241                    0  1180
Fiber optic      2257                    0   839
No                  0                 1526     0
MultipleLines    No  No phone service   Yes
PhoneService                               
No                0               682     0
Yes            3390                 0  2971


In [10]:
internet_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                 "TechSupport", "StreamingTV", "StreamingMovies"]

telco[internet_cols] = telco[internet_cols].replace("No internet service", "No")
telco["MultipleLines"] = telco["MultipleLines"].replace("No phone service", "No")

for c in internet_cols + ["MultipleLines"]:
    print(c, telco[c].unique())     # each should show only 'Yes' and 'No'

OnlineSecurity <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineBackup <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
DeviceProtection <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
TechSupport <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingMovies <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [11]:
print("duplicate rows:", telco.duplicated().sum())
print("duplicate customer IDs:", telco["customerID"].duplicated().sum())
telco = telco.drop(columns=["customerID"])

duplicate rows: 0
duplicate customer IDs: 0


In [12]:
print("missing values:", bank.isna().sum().sum())
print("duplicate rows:", bank.duplicated().sum())

bank = bank.drop(columns=["RowNumber", "CustomerId", "Surname"])
bank = bank.rename(columns={"Exited": "churn"})
print(bank["churn"].value_counts(normalize=True))

missing values: 0
duplicate rows: 0
churn
0    0.7963
1    0.2037
Name: proportion, dtype: float64


In [13]:
telco.to_csv("../data/processed/telco_clean.csv", index=False)
bank.to_csv("../data/processed/bank_clean.csv", index=False)
print(telco.shape, bank.shape)

(7043, 20) (10000, 11)
